# Train, evaluate and export a run

Run on a GPU runtime. Set the parameters in the next cell, then run the cells in order: setup, stage the shards and eval plots, throughput check, release baseline (once), train, evaluate, export, run comparison, close.

Re-running the train cell after a disconnect resumes from the last checkpoint on Drive.

The repo is public; this notebook still clones it with a read-only GitHub token from Colab Secrets (the key icon), named `GITHUB_TOKEN`, with notebook access on.

In [ ]:
RUN_ID = "neon-platform"
BASELINE_ID = "release-baseline"  # the released weights, scored once on the same benchmark
CONFIG = "experiments/configs/neon_platform.yaml"
REPO_SHA = ""
SOURCES = ["neon"]
WORKERS = 4  # loader workers: the fastest count in throughput.json (4 on an L4)

In [ ]:
import subprocess
import sys
from pathlib import Path

from google.colab import drive, userdata

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/fair_trees")

assert REPO_SHA, "set REPO_SHA to the pushed commit"
# a read-only GitHub token in Colab Secrets (key icon), named GITHUB_TOKEN
REPO = "github.com/OmdenaAI/HOTOSM-fAIr-Trees.git"
token = userdata.get("GITHUB_TOKEN")
!rm -rf /content/fair_trees
# git clone through subprocess so the token never reaches the cell output
url = f"https://x-access-token:{token}@{REPO}"
clone = subprocess.run(["git", "clone", "-q", url, "/content/fair_trees"])
# checked by hand: a CalledProcessError message would print the URL, token included
if clone.returncode:
    raise RuntimeError(f"git clone failed (exit {clone.returncode}): check the GITHUB_TOKEN secret")
# drop the token from the saved remote so it is not left in .git/config
git = ["git", "-C", "/content/fair_trees"]
subprocess.run([*git, "remote", "set-url", "origin", f"https://{REPO}"], check=True)
subprocess.run([*git, "checkout", "-q", REPO_SHA], check=True)
del token, url

%pip install "/content/fair_trees[train,test]" -r /content/fair_trees/experiments/requirements.txt
%cd /content/fair_trees
sys.path.insert(0, "/content/fair_trees")

In [ ]:
import json

import torch
import torchvision

from experiments import run

assert torch.__version__.split("+")[0] == "2.11.0" and torchvision.__version__.split("+")[0] == "0.26.0"
print(torch.cuda.get_device_name())

In [ ]:
import zipfile

from experiments.shards import stage_shards

stage_shards(DRIVE, Path("/content/data"), SOURCES)

# the eval plots come from the annotation and evaluation archives kept on Drive; only the labels
# and the RGB plots are extracted (evaluation.zip also holds the LiDAR)
parts = ("annotations", "evaluation/RGB")
raw_drive = DRIVE / "raw" / "neon"
listed = {f["name"] for f in json.loads((raw_drive / "manifest.json").read_text())["files"]}
eval_root = Path("/content/data/NeonTreeEvaluation")
eval_root.mkdir(parents=True, exist_ok=True)
for name in ("annotations.zip", "evaluation.zip"):
    assert name in listed, f"{name} is not in the raw manifest on Drive"
    with zipfile.ZipFile(raw_drive / name) as z:
        members = [m for m in z.namelist() if m.startswith(tuple(f"{p}/" for p in parts))]
        z.extractall(eval_root, members=members)
for part in parts:
    assert (eval_root / part).is_dir(), f"{eval_root / part} is missing after extraction"

In [ ]:
!python -m experiments.throughput $CONFIG $RUN_ID
throughput = json.loads((DRIVE / "runs" / RUN_ID / "throughput.json").read_text())
print(json.dumps(throughput, indent=2))
print("fastest loader workers:", max(throughput["loader"], key=throughput["loader"].get))
assert throughput["pass"], "loader is slower than the GPU step: stop here and report throughput.json"

Release baseline: run once. It scores the released weights on the same benchmark.

In [ ]:
run.evaluate(
    BASELINE_ID,
    Path("experiments/benchmark.json"),
    config=Path(CONFIG),
    weights=DRIVE / "weights" / "NEON.pt",
)

In [ ]:
result = run.main(Path(CONFIG), RUN_ID, num_workers=WORKERS)  # re-running this cell resumes
print(result.best_metrics)

In [ ]:
report = run.evaluate(RUN_ID, Path("experiments/benchmark.json"))
print("thresholds", report["thresholds"], "picked in", report["threshold_source"])
for name, part in report["parts"].items():  # val, val_group (two classes) or val_group_merged, NEON
    if "f1_ci95" not in part:  # val_in_group: a two-class model's tree boxes inside groups
        print(name, json.dumps(part))
        continue
    picked = f"| {part['f1_picked']:.3f} at {part['picked_threshold']}" if "f1_picked" in part else ""
    print(name, round(part["f1"], 3), [round(x, 3) for x in part["f1_ci95"]], picked)

In [ ]:
exported = run.export(RUN_ID, Path("experiments/benchmark.json"))
print(exported)
assert exported["pass"]

The resume check compares two runs of `experiments/configs/neon_resume_check.yaml`. To produce them:

1. In the parameters cell set `RUN_ID = "neon-resume-a"` and `CONFIG` to that file, run the setup, torch check, stage and train cells uninterrupted, then evaluate it.
2. Set `RUN_ID = "neon-resume-b"` and run the train cell. During epoch 1, once a mid-epoch checkpoint `ckpt-e001-s<step>.pt` with a nonzero step (not `ckpt-e001-s000000.pt`) appears in `runs/neon-resume-b` on Drive, disconnect and delete the runtime.
3. Reconnect and rerun, in order: the parameters cell (with `RUN_ID = "neon-resume-b"` and the same `CONFIG`), setup, torch check, stage, then the same train cell. Evaluate `neon-resume-b`.
4. Compare the two runs below.

In [ ]:
print(run.compare_runs("neon-resume-a", "neon-resume-b"))
print(run.compare_runs(BASELINE_ID, RUN_ID))

Each run at its own picked threshold (both runs need an `eval.json` from this code: re-run the evaluate cell for an older baseline first), and for a two-class run the group join rules from one decode of val.

In [ ]:
print(run.compare_runs(BASELINE_ID, RUN_ID, picked=True))
if "val_group" in report["parts"]:
    # a tree-only baseline has the crown-merge groups (at their own operating point) instead
    base_parts = json.loads((DRIVE / "runs" / BASELINE_ID / "eval.json").read_text())["parts"]
    part_a = "val_group" if "val_group" in base_parts else "val_group_merged"
    print(run.compare_runs(BASELINE_ID, RUN_ID, part_a=part_a, part_b="val_group", picked=True))
    merges = run.compare_group_merges(RUN_ID)
    for rule, r in merges["rules"].items():
        diff = [round(x, 3) for x in r["paired_diff_vs_join"]]
        print(rule, round(r["f1"], 3), "f1_max", round(r["f1_max"], 3), "vs join", diff)

Optional: outside weights on the same benchmark. Scores HOT's td-hot model (the release weights fine-tuned on Freetown with layer4 trainable; MIT, pinned to one Hugging Face commit) at our 60 m tree window and at 40 m, the 5 cm per pixel it was trained at, and re-scores `hot-platform` (the release weights fine-tuned on HOT with the backbone frozen) on today's HOT data, so every run shares one protocol. Needs `SOURCES` with `hot` and `hot-test`, and the `eval.json` of `hot-platform-joint0.3-newdata` and `hot-groups-crop80` on Drive. Runs once; each model is its own run folder.

In [ ]:
from experiments.import_weights import import_retinanet

NEON_OAM_URL = (
    "https://huggingface.co/AbdelrahmanU3/NEON_OAM/resolve/"
    "35cc87122b84ea40ec419455ddbff6aa1e552766/NEON_OAM.pt"
)
OUTSIDE_CONFIG = Path("experiments/configs/hot_platform_joint0.3.yaml")  # tree-only val on HOT at 60 m

raw = DRIVE / "weights" / "td-hot" / "NEON_OAM.pt"
if not raw.exists():
    raw.parent.mkdir(parents=True, exist_ok=True)
    torch.hub.download_url_to_file(NEON_OAM_URL, str(raw))
neon_oam = import_retinanet(raw, raw.with_name("NEON_OAM-frozen-bn.pt"))
outside = {
    "td-hot-neon-oam": (neon_oam, None),
    "td-hot-neon-oam-40m": (neon_oam, 40.0),
    "hot-platform-rescore": (DRIVE / "runs" / "hot-platform" / "best.pt", None),
}
for run_id, (weights, window) in outside.items():
    report = run.evaluate(
        run_id, Path("experiments/benchmark.json"), config=OUTSIDE_CONFIG, weights=weights, window_m=window
    )
    for name, part in report["parts"].items():
        if "f1_picked" in part:
            print(
                run_id,
                name,
                part.get("window_m", "NEON plots"),
                round(part["f1"], 3),
                f"| {part['f1_picked']:.3f} at {part['picked_threshold']}",
                f"P {part['precision_picked']:.3f} R {part['recall_picked']:.3f}",
            )

# b - a, each at its own picked threshold, paired over the same units (all at 60 m)
PAIRS = (
    ("hot-platform-rescore", "td-hot-neon-oam"),  # same start (release weights): his recipe vs ours
    ("hot-platform-rescore", "hot-platform-joint0.3-newdata"),  # what our joint base adds
    ("td-hot-neon-oam", "hot-platform-joint0.3-newdata"),  # his model vs our tree-only model
    ("td-hot-neon-oam", "hot-groups-crop80"),  # his model vs our current model (trees)
)
for a, b in PAIRS:
    for part in ("val", "hot-test"):
        r = run.compare_runs(a, b, part_a=part, picked=True)
        diff = [round(x, 3) for x in r["paired_diff"]]
        print(f"{b} - {a} on {part}: {r['f1_b']:.3f} vs {r['f1_a']:.3f}, paired {diff}")

Optional: where the missing tree recall goes, without training. For `RECALL_RUN` (a finished run on Drive), each setting is decoded once and scored at the run's picked tree threshold: boxes shrunk about their centres, recall per crown size, NMS thresholds and a 40 m tree window. Settings are chosen on val; held-out test is scored too, but only to confirm the val winner once. Writes `recall_check_<part>.json` in the run folder. Needs `SOURCES` with `hot` and `hot-test`.

In [ ]:
from experiments.recall_check import report_lines, run_recall_check

RECALL_RUN = "hot-groups-crop80"
for part in ("val", "hot-test"):
    print(part)
    print("\n".join(report_lines(run_recall_check(RECALL_RUN, part=part))))

Optional: are the missed small crowns trees? For `RECALL_RUN`, every truth crown under 2.5 m gets a row (matched at the picked threshold or not, best IoU with a kept box, top score of any box over its centre). A seeded sample of 30 misses, half under 1.5 m and half 1.5-2.5 m, and 12 hits for contrast are drawn on 12 m crops. Writes `small_crowns_<part>_misses.png`, `small_crowns_<part>_hits.png` and `small_crowns_<part>.json` (each tile's lon/lat) in the run folder. Download them to look at by eye.

In [ ]:
import json

from IPython.display import Image, display

from experiments.run import DRIVE_ROOT
from experiments.small_crowns import run_small_crowns

for part in ("val", "hot-test"):
    result = run_small_crowns(RECALL_RUN, part=part)
    print(part, json.dumps(result["by_size"], indent=1))
    display(Image(str(DRIVE_ROOT / "runs" / RECALL_RUN / f"small_crowns_{part}_misses.png")))

Optional: what a cheaper serving setting costs, without training. Serving time is the number of windows the network runs, which `window_overlap` sets. For `OVERLAP_RUN` (a finished run on Drive), each overlap's windows are run once per class and scored at every pick threshold under each edge band and, for tree groups, each merge rule. Per overlap the lines show the serving default at the run's picked threshold and, when one beats it, the best other setting, each paired against overlap 0.5; the overlap 0.5 line of each class repeats the evaluate cell's picked F1. Settings are chosen on val; held-out test is scored too, but only to confirm the val winner once. Writes `overlap_check_<part>.json` in the run folder. Needs `SOURCES` with `hot` and `hot-test`; runs right after the stage cell.

In [ ]:
import logging

from experiments.overlap_check import report_lines as overlap_lines
from experiments.overlap_check import run_overlap_check

logging.basicConfig(level=logging.WARNING)
logging.getLogger("experiments.overlap_check").setLevel(logging.INFO)  # a progress line per overlap

OVERLAP_RUN = "hot-groups-crop80"
for part in ("val", "hot-test"):
    print(part)
    print("\n".join(overlap_lines(run_overlap_check(OVERLAP_RUN, part=part))))

Optional: weight interpolation. Blends the base weights with this run's `best.pt` (`(1 - alpha) * base + alpha * tuned`) and scores each blend on this run's val regions and the NEON benchmark, to see whether a point between them keeps the val gain without the forgetting. Run after the evaluate cell; each blend is its own run folder `<RUN_ID>-wise-<alpha>`.

In [ ]:
from experiments.interpolate import interpolate

ALPHAS = (0.3, 0.5, 0.55, 0.6, 0.7)  # share of the fine-tuned weights
base_weights = DRIVE / run.load_config(Path(CONFIG))["weights"]
for alpha in ALPHAS:
    blend_id = f"{RUN_ID}-wise-{alpha:g}"
    blend = interpolate(
        base_weights, DRIVE / "runs" / RUN_ID / "best.pt", alpha, DRIVE / "runs" / blend_id / "weights.pt"
    )
    benchmark = Path("experiments/benchmark.json")
    blend_report = run.evaluate(blend_id, benchmark, config=Path(CONFIG), weights=blend)
    val, neon = blend_report["parts"]["val"], blend_report["parts"]["neon"]
    group = blend_report["parts"].get("val_group")
    groups = f", group F1 {group['f1']:.3f}" if group else ""
    gate = f"NEON F1 {neon['f1']:.3f} (pass {neon['forgetting_pass']})"
    print(f"{blend_id}: val F1 {val['f1']:.3f}{groups}, {gate}")
    print(run.compare_runs(RUN_ID, blend_id)["paired_diff"])

In [ ]:
drive.flush_and_unmount()